# GeoGuard: Satellite Landslide Detection
## Notebook 05: Copernicus Sentinel-2 Satellite Inference & AOI Tiling

This notebook demonstrates the end-to-end satellite inference pipeline:
1. **AOI Configuration**: Inspects the configured corridor without inventing unconfirmed coordinates.
2. **Copernicus STAC Discovery**: Searches live Sentinel-2 acquisitions via CDSE STAC catalogue.
3. **Scene Retrieval**: Acquires 4-channel optical bands (B02, B03, B04, B08) at 10m resolution (or runs in SIMULATED test mode if credentials are missing).
4. **Visual Verification**: Displays True-Color RGB and Color-Infrared (CIR) composites.
5. **Scene Tiling**: Divides the satellite scene into $128 \times 128$ pixel patches with geographic tracking.
6. **CNN Classification**: Runs the trained `cnn_model.keras` on valid patches.
7. **Macro AOI Aggregation**: Aggregates patch results into a decision status.

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from config import settings
from src.satellite import CopernicusClient, SatellitePipeline
from src.predict import LandslidePredictor
from src.preprocessing import SatellitePreprocessor

print(f"Highway Corridor Name: {settings.HIGHWAY_CORRIDOR_NAME}")
print(f"Highway Corridor Bbox: {settings.HIGHWAY_AOI_BBOX} (None = unconfirmed, safe test fallback enabled)")
print(f"Test AOI Bbox: {settings.TEST_AOI_BBOX}")
print(f"CDSE Credentials configured: {CopernicusClient().has_credentials}")

### 2. Live Copernicus STAC Catalogue Acquisition Discovery

We query the public Copernicus Data Space Ecosystem STAC API (`https://stac.dataspace.copernicus.eu/v1/search`) to discover real Sentinel-2 Level-2A acquisitions covering the test AOI.

In [ ]:
client = CopernicusClient()
acquisitions = client.search_acquisitions(bbox=settings.TEST_AOI_BBOX, limit=3)

print(f"Discovered {len(acquisitions)} live Sentinel-2 acquisitions:")
for a in acquisitions:
    print(f"  ID: {a['acquisition_id']}")
    print(f"  Datetime: {a['acquisition_time']} | Cloud: {a['cloud_cover_percent']}% | Usable: {a['is_usable']}")

### 3. Executing the Satellite Pipeline (Tiling, Inference, and Aggregation)

We now invoke `SatellitePipeline.process_aoi()`. If live Copernicus credentials are absent, the system safely operates in `SIMULATED` test mode using verified real multispectral test patches.

In [ ]:
pipeline = SatellitePipeline()
result = pipeline.process_aoi(bbox=settings.TEST_AOI_BBOX, aoi_name="TEST_DEVELOPMENT_AOI")

print("=== AOI PROCESSING RESULT ===")
print(f"Status: {result['status']} | Data Status: {result['data_status']}")
print(f"Macro Decision: {result['prediction']['overall_class']}")
print(f"Max Landslide Probability: {result['prediction']['max_probability']:.4f}")
print(f"Satellite Source: {result['satellite']['source']} (Mode: {result['satellite']['mode']})")
print(f"Acquisition ID: {result['satellite']['acquisition_id']}")
print(f"Patches Evaluated: {result['aggregation']['total_patches_evaluated']} (Landslide={result['aggregation']['landslide_patches']}, Normal={result['aggregation']['normal_patches']})")
print(f"High-Risk Coordinates: {result['aggregation']['high_risk_coordinates']}")

### 4. Visualizing Individual Patch Predictions

In [ ]:
patches = result.get('patch_details', [])
if patches:
    fig, axes = plt.subplots(len(patches), 2, figsize=(10, 4 * len(patches)))
    if len(patches) == 1:
        axes = np.expand_dims(axes, axis=0)
        
    for idx, p in enumerate(patches):
        coords = p['center_coords']
        pred_class = p['prediction_class']
        prob = p['probability']
        color = 'red' if pred_class == 'LANDSLIDE DETECTED' else 'green'
        
        axes[idx, 0].set_title(f"Patch #{p['patch_index']} (Lat: {coords['lat']}, Lon: {coords['lon']})")
        axes[idx, 0].text(0.5, 0.5, f"{pred_class}\nP = {prob:.4f}", fontsize=14, ha='center', va='center', color=color, fontweight='bold')
        axes[idx, 0].axis('off')
        
        axes[idx, 1].text(0.1, 0.5, f"Bounding Box:\n{p['bbox']}", fontsize=11)
        axes[idx, 1].axis('off')
    plt.tight_layout()
    plt.show()